<a href="https://colab.research.google.com/github/AlperYildirim1/geometric-grokking/blob/main/Final_Modular_Norm_Baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import json

SAVE_DIR = '/content/drive/MyDrive/grokking_layernorm_models_6e-4'
os.makedirs(SAVE_DIR, exist_ok=True)

# CONFIGURATION
P = 113
FRAC_TRAIN = 0.3
D_MODEL = 128
NUM_HEADS = 4
MLP_DIM = 512
LR = 6e-4 # Change to 1e-4 for other results.
WEIGHT_DECAY = 1.0
LOG_EVERY = 200
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DETERMINISM
def set_seed(seed):
    """Locks down all sources of randomness for 100% reproducibility."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

# DATASET GENERATION
def make_dataset(p, frac_train, seed=42):
    set_seed(seed)
    rng = random.Random(seed)
    all_pairs = [(a, b) for a in range(p) for b in range(p)]
    rng.shuffle(all_pairs)

    n_train = int(len(all_pairs) * frac_train)
    train_x = torch.tensor([[a, b, p] for a, b in all_pairs[:n_train]], dtype=torch.long)
    train_y = torch.tensor([(a + b) % p for a, b in all_pairs[:n_train]], dtype=torch.long)
    test_x = torch.tensor([[a, b, p] for a, b in all_pairs[n_train:]], dtype=torch.long)
    test_y = torch.tensor([(a + b) % p for a, b in all_pairs[n_train:]], dtype=torch.long)
    return train_x, train_y, test_x, test_y

# 1. RMSNORM IMPLEMENTATION
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        # The learnable scale parameter (gamma) allows magnitude to explode
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        norm_x = torch.mean(x ** 2, dim=-1, keepdim=True)
        return x * torch.rsqrt(norm_x + self.eps) * self.weight

# 2. STANDARD TRANSFORMER WITH NORM OPTIONS
class StandardTransformer(nn.Module):
    def __init__(self, p, d_model, num_heads, mlp_dim, norm_type='none'):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.norm_type = norm_type

        self.tok_embed = nn.Embedding(p + 1, d_model)
        self.pos_embed = nn.Embedding(3, d_model)

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

        self.mlp_in = nn.Linear(d_model, mlp_dim)
        self.mlp_out = nn.Linear(mlp_dim, d_model)

        # Standard unconstrained unembedding
        self.unembed = nn.Linear(d_model, p, bias=False)

        if norm_type == 'layernorm':
            self.norm1 = nn.LayerNorm(d_model)
            self.norm2 = nn.LayerNorm(d_model)
            self.norm3 = nn.LayerNorm(d_model)
        elif norm_type == 'rmsnorm':
            self.norm1 = RMSNorm(d_model)
            self.norm2 = RMSNorm(d_model)
            self.norm3 = RMSNorm(d_model)

    def apply_norm(self, x, norm_module=None):
        if self.norm_type in ['layernorm', 'rmsnorm'] and norm_module is not None:
            return norm_module(x)
        return x

    def forward(self, x):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h = self.tok_embed(x) + self.pos_embed(pos)

        h = self.apply_norm(h, getattr(self, 'norm1', None))

        Q = self.W_Q(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        K = self.W_K(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        V = self.W_V(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn_out = self.W_O((F.softmax(scores, dim=-1) @ V).transpose(1, 2).contiguous().view(B, L, self.d_model))

        h = h + attn_out
        h = self.apply_norm(h, getattr(self, 'norm2', None))

        h = h + self.mlp_out(F.relu(self.mlp_in(h)))
        h = self.apply_norm(h, getattr(self, 'norm3', None))

        return self.unembed(h[:, 2, :])

# 3. TRAINING LOGIC
def train_model(model, name, train_x, train_y, test_x, test_y, epochs):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999))
    criterion = nn.CrossEntropyLoss()

    train_x, train_y = train_x.to(DEVICE), train_y.to(DEVICE)
    test_x, test_y = test_x.to(DEVICE), test_y.to(DEVICE)

    grok_epoch = None
    history = {'epochs': [], 'train_acc': [], 'test_acc': [], 'train_loss': [], 'test_loss': []}

    pbar = tqdm(range(epochs), desc=f"Training {name}")

    for epoch in pbar:
        model.train()
        logits = model(train_x)
        loss = criterion(logits, train_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % LOG_EVERY == 0 or epoch == epochs - 1:
            model.eval()
            with torch.no_grad():
                train_acc = (logits.argmax(-1) == train_y).float().mean().item()
                test_logits = model(test_x)
                test_loss = criterion(test_logits, test_y).item()
                test_acc = (test_logits.argmax(-1) == test_y).float().mean().item()

            history['epochs'].append(epoch)
            history['train_acc'].append(train_acc)
            history['test_acc'].append(test_acc)
            history['train_loss'].append(loss.item())
            history['test_loss'].append(test_loss)

            pbar.set_postfix({'tr_acc': f"{train_acc:.3f}", 'te_acc': f"{test_acc:.3f}"})

            if grok_epoch is None and test_acc > 0.95:
                grok_epoch = epoch
                tqdm.write(f"\n⚡ {name} generalized at epoch {epoch}! (test_acc={test_acc:.4f})")

    return {"grok_epoch": grok_epoch if grok_epoch else f">{epochs}", "history": history}

# EXECUTION & PLOTTING
if __name__ == "__main__":
    print("=" * 60)
    print("Z_113 NORMALIZATION ABLATION: LayerNorm vs RMSNorm")
    print("=" * 60)

    # 10 seeds
    seeds_to_run = list(range(1, 11))

    for SEED in seeds_to_run:
        print(f"\n\n{'=' * 40}")
        print(f" STARTING RUN FOR SEED {SEED}")
        print(f"{'=' * 40}\n")

        # Generate Z_113 Dataset
        tr_x, tr_y, te_x, te_y = make_dataset(P, FRAC_TRAIN, seed=SEED)

        # Initialize Models
        models = [
            ("LayerNorm", StandardTransformer(P, D_MODEL, NUM_HEADS, MLP_DIM, norm_type='layernorm'), 15000),
            ("RMSNorm", StandardTransformer(P, D_MODEL, NUM_HEADS, MLP_DIM, norm_type='rmsnorm'), 15000)
        ]

        results = {}
        for name, model, train_epochs in models:
            set_seed(SEED)
            model = model.to(DEVICE)
            results[name] = train_model(model, name, tr_x, tr_y, te_x, te_y, train_epochs)

            safe_name = f"Z113_{name.replace(' ', '_')}"
            save_path = os.path.join(SAVE_DIR, f"{safe_name}_seed{SEED}.json")
            with open(save_path, "w") as f:
                json.dump(results[name]["history"], f)

        # Plotting 1x2 Grid
        fig_1x2, axs_1x2 = plt.subplots(1, 2, figsize=(12, 5))
        axs_1x2 = axs_1x2.flatten()

        for i, (name, res) in enumerate(results.items()):
            hist = res["history"]
            axs_1x2[i].plot(hist["epochs"], hist["train_acc"], label="Train Acc", color="#1f77b4", linewidth=2.5)
            axs_1x2[i].plot(hist["epochs"], hist["test_acc"], label="Test Acc", color="#d62728", linewidth=2.5)
            axs_1x2[i].fill_between(hist["epochs"], hist["train_acc"], hist["test_acc"], color='grey', alpha=0.15)

            title_text = f"Z_113 {name}\nGeneralized at: {res['grok_epoch']}"
            axs_1x2[i].set_title(title_text, fontsize=16, pad=10)
            axs_1x2[i].set_xlabel("Epochs", fontsize=14)
            axs_1x2[i].set_ylabel("Accuracy", fontsize=14)
            axs_1x2[i].set_xlim(0, 15000)
            axs_1x2[i].tick_params(axis='both', which='major', labelsize=12)
            axs_1x2[i].grid(True, alpha=0.3)
            axs_1x2[i].legend(loc="lower right", fontsize=12)

        plt.tight_layout()
        plot_path = os.path.join(SAVE_DIR, f"z113_norm_ablation_grid_seed{SEED}.png")
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        print(f" Saved plot to: {plot_path}")
        plt.close(fig_1x2)

In [ ]:
import os
import json
import glob
import pandas as pd
import numpy as np

# The two directories we want to analyze
dirs_to_check = [
    '/content/drive/MyDrive/grokking_layernorm_models_1e-4',
    '/content/drive/MyDrive/grokking_layernorm_models_6e-4'
]

for directory in dirs_to_check:
    print(f"\n{'='*70}")
    print(f"📊 ANALYZING: {os.path.basename(directory)}")
    print(f"{'='*70}")

    if not os.path.exists(directory):
        print(f" Directory not found: {directory}\nMake sure Google Drive is mounted!")
        continue

    # Find all JSON files
    json_files = glob.glob(os.path.join(directory, "*.json"))

    if not json_files:
        print(f" No JSON results found in this directory.")
        continue

    results = []
    for filepath in json_files:
        filename = os.path.basename(filepath)
        # Parse architecture from filename (e.g., "Z113_LayerNorm_seed1.json" -> "LayerNorm")
        arch_name = filename.split('_seed')[0].replace('Z113_', '')

        with open(filepath, 'r') as f:
            hist = json.load(f)

        # Find the grokking epoch (> 95% test accuracy)
        epochs = hist['epochs']
        test_accs = hist['test_acc']

        grok_epoch = None
        for ep, acc in zip(epochs, test_accs):
            if acc > 0.95:
                grok_epoch = ep
                break

        peak_acc = max(test_accs)

        results.append({
            'Architecture': arch_name,
            'Grok_Epoch': grok_epoch if grok_epoch is not None else np.nan,
            'Peak_Test_Acc': peak_acc
        })

    df = pd.DataFrame(results)

    if not df.empty:
        # Calculate stats
        summary = df.groupby('Architecture').agg(
            Mean_Grok_Epoch=('Grok_Epoch', 'mean'),
            Std_Grok_Epoch=('Grok_Epoch', 'std'),
            Min_Grok_Epoch=('Grok_Epoch', 'min'),
            Max_Grok_Epoch=('Grok_Epoch', 'max'),
            Did_Not_Grok=('Grok_Epoch', lambda x: x.isna().sum()),
            Mean_Peak_Acc=('Peak_Test_Acc', 'mean')
        ).reset_index()

        pd.set_option('display.float_format', lambda x: '%.2f' % x)
        print(summary.to_string(index=False))